In [2]:
import requests
import time
import re
import sqlite3
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [3]:
BASE_URL  = "https://hubeau.eaufrance.fr/api/v1/qualite_eau_potable/resultats_dis"
DB_PATH   = "eau_potable.db"

VILLES = {
    "Lyon":          "69123",
    "Saint-Etienne": "42218",
}

DATE_MIN   = "2025-01-01"
TIMEOUT    = 30
SLEEP      = 1.0
MAX_RETRY  = 3

# Colonnes brutes demandées à l'API
FIELDS_API = ",".join([
    "date_prelevement",
    "code_commune",
    "nom_commune",
    "nom_departement",
    "nom_distributeur",
    "nom_installation_amont",
    "libelle_parametre",
    "code_parametre",
    "code_type_parametre",
    "resultat_numerique",
    "resultat_alphanumerique",
    "libelle_unite",
    "limite_qualite_parametre",
    "reference_qualite_parametre",
    "conformite_limites_pc_prelevement",
    "conformite_references_pc_prelevement",
    "conclusion_conformite_prelevement",
])

In [4]:
def build_session() -> requests.Session:
    session = requests.Session()
    retry = Retry(
        total=MAX_RETRY,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504]
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    return session

SESSION = build_session()

In [5]:
def extract_ville(code_commune: str, nom_ville: str, date_min: str) -> list:
    records, page = [], 1

    while True:
        params = {
            "code_commune":          code_commune,
            "date_min_prelevement":  date_min,
            "size":                  200,
            "page":                  page,
            "fields":                FIELDS_API,
        }

        for attempt in range(1, MAX_RETRY + 1):
            try:
                response = SESSION.get(BASE_URL, params=params, timeout=TIMEOUT)
                response.raise_for_status()
                break
            except requests.exceptions.ReadTimeout:
                print(f"  Timeout page {page} — tentative {attempt}/{MAX_RETRY}")
                if attempt == MAX_RETRY:
                    raise
                time.sleep(5 * attempt)

        body        = response.json()
        page_records = body.get("data", [])

        for r in page_records:
            r["nom_ville"] = nom_ville

        records.extend(page_records)
        print(f"{nom_ville} — page {page} : {len(page_records)} résultats")

        if not body.get("next"):
            break

        page += 1
        time.sleep(SLEEP)

    return records

In [6]:
def extract(villes: dict, date_min: str) -> list:
    all_records = []
    for nom, code in villes.items():
        records = extract_ville(code, nom, date_min)
        all_records.extend(records)
        print(f"→ {nom} : {len(records)} enregistrements\n")
    print(f"Total brut : {len(all_records)} enregistrements")
    return all_records

raw_data = extract(VILLES, DATE_MIN)

Lyon — page 1 : 200 résultats
Lyon — page 2 : 200 résultats
Lyon — page 3 : 200 résultats
Lyon — page 4 : 200 résultats
Lyon — page 5 : 200 résultats
Lyon — page 6 : 200 résultats
Lyon — page 7 : 200 résultats
Lyon — page 8 : 200 résultats
Lyon — page 9 : 200 résultats
Lyon — page 10 : 200 résultats
Lyon — page 11 : 200 résultats
Lyon — page 12 : 200 résultats
Lyon — page 13 : 200 résultats
Lyon — page 14 : 200 résultats
Lyon — page 15 : 200 résultats
Lyon — page 16 : 200 résultats
Lyon — page 17 : 200 résultats
Lyon — page 18 : 200 résultats
Lyon — page 19 : 200 résultats
Lyon — page 20 : 200 résultats
Lyon — page 21 : 200 résultats
Lyon — page 22 : 200 résultats
Lyon — page 23 : 200 résultats
Lyon — page 24 : 200 résultats
Lyon — page 25 : 200 résultats
Lyon — page 26 : 200 résultats
Lyon — page 27 : 200 résultats
Lyon — page 28 : 200 résultats
Lyon — page 29 : 200 résultats
Lyon — page 30 : 200 résultats
Lyon — page 31 : 200 résultats
Lyon — page 32 : 200 résultats
Lyon — page 33 : 

In [7]:
def load_bronze(records: list, db_path: str) -> None:
    df = pd.DataFrame(records)
    conn = sqlite3.connect(db_path)

    # Table stations : une ligne par installation unique
    bronze_stations = df[[
        "code_commune", "nom_commune", "nom_departement",
        "nom_distributeur", "nom_installation_amont", "nom_ville"
    ]].drop_duplicates()

    bronze_stations.to_sql("bronze_stations", conn, if_exists="replace", index=False)

    # Table paramètres : une ligne par paramètre unique
    bronze_parametres = df[[
        "code_parametre", "libelle_parametre",
        "code_type_parametre", "libelle_unite"
    ]].drop_duplicates()

    bronze_parametres.to_sql("bronze_parametres", conn, if_exists="replace", index=False)

    # Table analyses : toutes les mesures brutes
    bronze_analyses = df[[
        "nom_ville", "code_commune", "date_prelevement",
        "code_parametre", "libelle_parametre",
        "resultat_numerique", "resultat_alphanumerique", "libelle_unite",
        "limite_qualite_parametre", "reference_qualite_parametre",
        "conformite_limites_pc_prelevement",
        "conformite_references_pc_prelevement",
        "conclusion_conformite_prelevement",
    ]]

    bronze_analyses.to_sql("bronze_analyses", conn, if_exists="replace", index=False)

    conn.close()
    print(f"Bronze chargé — {len(bronze_stations)} stations | "
          f"{len(bronze_parametres)} paramètres | "
          f"{len(bronze_analyses)} analyses")

load_bronze(raw_data, DB_PATH)

Bronze chargé — 8 stations | 592 paramètres | 27089 analyses


In [8]:
conn = sqlite3.connect(DB_PATH)
print("=== bronze_stations ===")
display(pd.read_sql("SELECT * FROM bronze_stations LIMIT 5", conn))

print("=== bronze_parametres ===")
display(pd.read_sql("SELECT * FROM bronze_parametres LIMIT 5", conn))

print("=== bronze_analyses ===")
display(pd.read_sql("SELECT * FROM bronze_analyses LIMIT 5", conn))
conn.close()

=== bronze_stations ===


,code_commune,nom_commune,nom_departement,nom_distributeur,nom_installation_amont,nom_ville
0,69123,LYON,Rhône,EAU DU GRAND LYON,CENTRE (METROPOLE LYON),Lyon
1,42218,SAINT-ETIENNE,Loire,SAUR,SEM FURAN SOLAURE,Saint-Etienne
2,42218,SAINT-ETIENNE,Loire,SAUR,None,Saint-Etienne
3,42218,SAINT-ETIENNE,Loire,SAUR,SAINT ETIENNE SOLAURE,Saint-Etienne
4,42218,SAINT-ETIENNE,Loire,SAUR,ROCHETAILLEE CHLORATION,Saint-Etienne


=== bronze_parametres ===


,code_parametre,libelle_parametre,code_type_parametre,libelle_unite
0,1295,Turbidité néphélométrique NFU,N,NFU
1,5901,Odeur (qualitatif),O,SANS OBJET
2,1374,Calcium,N,mg/L
3,1398,Chlore libre,N,mg(Cl2)/L
4,1335,Ammonium (en NH4),N,mg/L


=== bronze_analyses ===


,nom_ville,code_commune,date_prelevement,code_parametre,libelle_parametre,resultat_numerique,resultat_alphanumerique,libelle_unite,limite_qualite_parametre,reference_qualite_parametre,conformite_limites_pc_prelevement,conformite_references_pc_prelevement,conclusion_conformite_prelevement
0,Lyon,69123,2026-02-25T09:27:00Z,1295,Turbidité néphélométrique NFU,0.12,"0,12",NFU,None,<=2 NFU,C,C,Eau d'alimentation conforme aux exigences de q...
1,Lyon,69123,2026-02-25T09:27:00Z,5901,Odeur (qualitatif),0.00,Aucun changement anormal,SANS OBJET,None,None,C,C,Eau d'alimentation conforme aux exigences de q...
2,Lyon,69123,2026-02-25T09:27:00Z,1374,Calcium,69.00,"69,0",mg/L,None,None,C,C,Eau d'alimentation conforme aux exigences de q...
3,Lyon,69123,2026-02-25T09:27:00Z,1398,Chlore libre,0.15,"0,15",mg(Cl2)/L,None,None,C,C,Eau d'alimentation conforme aux exigences de q...
4,Lyon,69123,2026-02-25T09:27:00Z,1335,Ammonium (en NH4),0.00,"<0,05",mg/L,None,"<=0,1 mg/L",C,C,Eau d'alimentation conforme aux exigences de q...


In [9]:
def extract_limite(raw: str):
    """'<=0,1 µg/L' → (0.1, 'µg/L')"""
    if pd.isna(raw):
        return float("nan"), None
    cleaned = str(raw).replace("<=", "").replace("<", "").strip()
    match   = re.search(r"([\d]+[,.]?[\d]*)\s*(.*)", cleaned)
    if match:
        val  = float(match.group(1).replace(",", "."))
        unit = match.group(2).strip() or None
        return val, unit
    return float("nan"), None


def flag_outlier(series: pd.Series, factor: float = 3.0) -> pd.Series:
    """True si la valeur dépasse factor * IQR au-dessus de Q3"""
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr    = q3 - q1
    return series > (q3 + factor * iqr)

In [10]:
def build_silver_stations(db_path: str) -> pd.DataFrame:
    conn = sqlite3.connect(db_path)
    df   = pd.read_sql("SELECT * FROM bronze_stations", conn)
    conn.close()

    # Suppression des doublons stricts
    df = df.drop_duplicates()

    # Nettoyage des noms : strip + title case
    for col in ["nom_commune", "nom_departement", "nom_distributeur", "nom_installation_amont"]:
        df[col] = df[col].str.strip().str.title()

    # Vérification intégrité : aucun code_commune null
    n_null = df["code_commune"].isna().sum()
    if n_null > 0:
        print(f"  Attention : {n_null} lignes sans code_commune supprimées")
        df = df.dropna(subset=["code_commune"])

    return df.reset_index(drop=True)

silver_stations = build_silver_stations(DB_PATH)
print(f"Silver stations : {len(silver_stations)} lignes")
display(silver_stations.head())

Silver stations : 8 lignes


,code_commune,nom_commune,nom_departement,nom_distributeur,nom_installation_amont,nom_ville
0,69123,Lyon,Rhône,Eau Du Grand Lyon,Centre (Metropole Lyon),Lyon
1,42218,Saint-Etienne,Loire,Saur,Sem Furan Solaure,Saint-Etienne
2,42218,Saint-Etienne,Loire,Saur,None,Saint-Etienne
3,42218,Saint-Etienne,Loire,Saur,Saint Etienne Solaure,Saint-Etienne
4,42218,Saint-Etienne,Loire,Saur,Rochetaillee Chloration,Saint-Etienne


In [11]:
def build_silver_analyses(db_path: str) -> pd.DataFrame:
    conn = sqlite3.connect(db_path)
    df   = pd.read_sql("SELECT * FROM bronze_analyses", conn)
    conn.close()

    # Typage
    df["date_prelevement"] = pd.to_datetime(df["date_prelevement"], errors="coerce")
    df["resultat_numerique"] = pd.to_numeric(df["resultat_numerique"], errors="coerce")

    # Extraction limite numérique + unité
    limites            = df["limite_qualite_parametre"].apply(extract_limite)
    df["limite_val"]   = limites.apply(lambda x: x[0])
    df["limite_unite"] = limites.apply(lambda x: x[1])

    # Vérification cohérence des unités
    df["unite_coherente"] = df["libelle_unite"] == df["limite_unite"]

    # Suppression des lignes sans mesure numérique ET sans mesure qualitative
    df = df.dropna(subset=["resultat_numerique"])

    # Doublons : même prélèvement / paramètre / commune
    n_before = len(df)
    df = df.drop_duplicates(subset=["code_commune", "date_prelevement", "code_parametre"])
    print(f"  Doublons supprimés : {n_before - len(df)}")

    # Outliers par paramètre
    df["outlier"] = df.groupby("libelle_parametre")["resultat_numerique"].transform(flag_outlier)

    # Conformité standardisée
    df["conforme"] = df["conformite_limites_pc_prelevement"].map({
        "C": True, "N": False, "S": None
    })

    # Ratio valeur / limite (uniquement si unités cohérentes)
    df["ratio_limite"] = (df["resultat_numerique"] / df["limite_val"]).where(
        df["unite_coherente"]
    ).round(3)

    cols = [
        "nom_ville", "code_commune", "date_prelevement",
        "code_parametre", "libelle_parametre",
        "resultat_numerique", "libelle_unite",
        "limite_val", "limite_unite", "unite_coherente",
        "ratio_limite", "conforme", "outlier",
    ]
    return df[cols].sort_values("date_prelevement").reset_index(drop=True)

silver_analyses = build_silver_analyses(DB_PATH)
print(f"Silver analyses : {len(silver_analyses)} lignes")
display(silver_analyses.head(10))

  Doublons supprimés : 60
Silver analyses : 26241 lignes


,nom_ville,code_commune,date_prelevement,code_parametre,libelle_parametre,resultat_numerique,libelle_unite,limite_val,limite_unite,unite_coherente,ratio_limite,conforme,outlier
0,Lyon,69123,2025-01-02 09:35:00+00:00,1302,pH,7.10,unité pH,NaN,None,False,NaN,True,False
1,Lyon,69123,2025-01-02 09:35:00+00:00,6455,Entérocoques /100ml-MS,0.00,n/(100mL),0.0,n/(100mL),True,NaN,True,False
2,Lyon,69123,2025-01-02 09:35:00+00:00,1295,Turbidité néphélométrique NFU,0.00,NFU,NaN,None,False,NaN,True,False
3,Lyon,69123,2025-01-02 09:35:00+00:00,1303,Conductivité à 25°C,411.00,µS/cm,NaN,None,False,NaN,True,False
4,Lyon,69123,2025-01-02 09:35:00+00:00,5441,Bact. aér. revivifiables à 36°-44h,0.00,n/mL,NaN,None,False,NaN,True,False
5,Lyon,69123,2025-01-02 09:35:00+00:00,1042,Bact. et spores sulfito-rédu./100ml,0.00,n/(100mL),NaN,None,False,NaN,True,False
6,Lyon,69123,2025-01-02 09:35:00+00:00,5440,Bact. aér. revivifiables à 22°-68h,0.00,n/mL,NaN,None,False,NaN,True,False
7,Lyon,69123,2025-01-02 09:35:00+00:00,1398,Chlore libre,0.15,mg(Cl2)/L,NaN,None,False,NaN,True,False
8,Lyon,69123,2025-01-02 09:35:00+00:00,1304,Conductivité à 20°C,370.00,µS/cm,NaN,None,False,NaN,True,False
9,Lyon,69123,2025-01-02 09:35:00+00:00,5901,Odeur (qualitatif),0.00,SANS OBJET,NaN,None,False,NaN,True,False


In [12]:
def build_silver_conformite(silver_analyses: pd.DataFrame) -> pd.DataFrame:
    return (
        silver_analyses
        .groupby(["nom_ville", "code_commune", "libelle_parametre"], as_index=False)
        .agg(
            nb_mesures      = ("resultat_numerique", "count"),
            nb_conformes    = ("conforme", lambda x: x.eq(True).sum()),
            nb_non_conformes= ("conforme", lambda x: x.eq(False).sum()),
            valeur_moyenne  = ("resultat_numerique", "mean"),
            valeur_max      = ("resultat_numerique", "max"),
            limite          = ("limite_val", "first"),
        )
        .assign(
            taux_conformite=lambda df: (
                df["nb_conformes"] / df["nb_mesures"] * 100
            ).round(1)
        )
        .sort_values("taux_conformite")
    )

silver_conformite = build_silver_conformite(silver_analyses)
print(f"Silver conformité : {len(silver_conformite)} lignes")
display(silver_conformite.head(10))

Silver conformité : 634 lignes


,nom_ville,code_commune,libelle_parametre,nb_mesures,nb_conformes,nb_non_conformes,valeur_moyenne,valeur_max,limite,taux_conformite
623,Saint-Etienne,42218,Tébutam,18,18,0,0.000000,0.00,0.1,100.0
622,Saint-Etienne,42218,Tébufénozide,17,17,0,0.000000,0.00,0.1,100.0
621,Saint-Etienne,42218,Tébuconazole,18,18,0,0.000000,0.00,0.1,100.0
620,Saint-Etienne,42218,Turbidité néphélométrique NFU,390,390,0,0.137231,1.30,1.0,100.0
619,Saint-Etienne,42218,Tritosulfuron,17,17,0,0.000000,0.00,0.1,100.0
618,Saint-Etienne,42218,Trinéxapac-éthyl,1,1,0,0.000000,0.00,0.1,100.0
617,Saint-Etienne,42218,Trimethacarbe,17,17,0,0.000000,0.00,0.1,100.0
616,Saint-Etienne,42218,Trihalométhanes (4 substances),35,35,0,14.848000,36.94,100.0,100.0
615,Saint-Etienne,42218,Trifluraline,18,18,0,0.000000,0.00,0.1,100.0
614,Saint-Etienne,42218,Triflumuron,1,1,0,0.000000,0.00,0.1,100.0


In [13]:
conn = sqlite3.connect(DB_PATH)
silver_stations.to_sql("silver_stations",    conn, if_exists="replace", index=False)
silver_analyses.to_sql("silver_analyses",    conn, if_exists="replace", index=False)
silver_conformite.to_sql("silver_conformite", conn, if_exists="replace", index=False)
conn.close()
print("Silver sauvegardé.")

Silver sauvegardé.


In [ ]:
def build_gold_dimensions(db_path: str) -> tuple:
    conn = sqlite3.connect(db_path)
    stations  = pd.read_sql("SELECT * FROM silver_stations", conn)
    analyses  = pd.read_sql("SELECT * FROM silver_analyses", conn)
    conn.close()

    # Dimension stations
    dim_stations = stations[[
        "code_commune", "nom_commune", "nom_departement",
        "nom_distributeur", "nom_installation_amont", "nom_ville"
    ]].drop_duplicates().reset_index(drop=True)
    dim_stations.index.name = "id_station"

    # Dimension paramètres
    dim_parametres = (
        analyses[["code_parametre", "libelle_parametre", "libelle_unite"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    dim_parametres.index.name = "id_parametre"

    # Dimension temps
    dim_temps = pd.DataFrame({
        "date":     pd.to_datetime(analyses["date_prelevement"]).dt.normalize().unique()
    }).sort_values("date").reset_index(drop=True)
    dim_temps["annee"]    = dim_temps["date"].dt.year
    dim_temps["mois"]     = dim_temps["date"].dt.month
    dim_temps["trimestre"]= dim_temps["date"].dt.quarter
    dim_temps.index.name  = "id_temps"

    return dim_stations, dim_parametres, dim_temps

dim_stations, dim_parametres, dim_temps = build_gold_dimensions(DB_PATH)
print(f"dim_stations   : {len(dim_stations)} lignes")
print(f"dim_parametres : {len(dim_parametres)} lignes")
print(f"dim_temps      : {len(dim_temps)} lignes")

AttributeError: 'Series' object has no attribute 'to_datetime'